# Phase 2: Lag-feature model and rolling backtest

Lag features (e.g. price lags 1–30, optional log returns), ColumnTransformer + Pipeline, rolling backtest, TimeSeriesSplit + RandomizedSearchCV, and evaluation (MAE, RMSE, directional accuracy, residual plots).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
elif ROOT.name != 'crypto-price-prediction':
    ROOT = ROOT / 'crypto-price-prediction'
sys.path.insert(0, str(ROOT))
from src.metrics import regression_metrics

## 1. Load data and time-based split (same as 01)

In [ ]:
DATA_DIR = ROOT / 'data'
df = pd.read_parquet(DATA_DIR / 'BTC_USD_daily.parquet')
n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]
print('Train', train.index[0], '->', train.index[-1], len(train))
print('Val  ', val.index[0], '->', val.index[-1], len(val))
print('Test ', test.index[0], '->', test.index[-1], len(test))

## 2.1 Feature design: lag features (and optional log returns)

**Lags:** price at t-1, t-2, …, t-`n_lags` (e.g. 7 or 30).

**Optional:** log return = log(price_t / price_{t-1}); can add lagged log returns as features.

Build a feature matrix where each row is one day; target = next-day price (or next-day return).

In [ ]:
N_LAGS = 30  # e.g. 7 or 30
price = df['price'].values

def build_lag_features(price, n_lags, add_log_return=False):
    """X: (T - n_lags) x n_lags (and optionally extra cols); y: next-day price."""
    T = len(price)
    X_list = []
    for lag in range(1, n_lags + 1):
        X_list.append(price[n_lags - lag : T - lag])
    X = np.column_stack(X_list)  # each row = [p(t-30), ..., p(t-1)]
    y = price[n_lags:-1]  # next-day price for each row (p(t) then p(t+1) ...)
    # Actually: row i corresponds to day index (n_lags + i); target = price at (n_lags + i + 1)
    y = price[n_lags + 1 :]
    X = X[:-1]
    if add_log_return:
        log_ret = np.log(price[1:] / price[:-1])
        lag_ret = [log_ret[n_lags - lag : T - lag - 1] for lag in range(1, min(8, n_lags) + 1)]
        X_ret = np.column_stack(lag_ret)
        X = np.hstack([X, X_ret])
    return X, y

X_train, y_train = build_lag_features(train['price'].values, N_LAGS)
X_val, y_val = build_lag_features(val['price'].values, N_LAGS)
X_test, y_test = build_lag_features(test['price'].values, N_LAGS)
print('X_train', X_train.shape, 'y_train', y_train.shape)

## 2.2 Preprocessing + Pipeline

ColumnTransformer(StandardScaler) + Ridge (or GradientBoostingRegressor). Same train/val/test alignment as above.

In [ ]:
n_features = X_train.shape[1]
ct = ColumnTransformer(
    [('scale', StandardScaler(), list(range(n_features)))],
    remainder='passthrough'
)
pipe = Pipeline([
    ('preprocess', ct),
    ('regressor', Ridge(alpha=1.0))
])
pipe.fit(X_train, y_train)
pred_val = pipe.predict(X_val)
pred_test = pipe.predict(X_test)
m_val = regression_metrics(y_val, pred_val)
m_test = regression_metrics(y_test, pred_test)
print('Val  ', m_val)
print('Test ', m_test)

## 2.3 Rolling backtest

Train on past only, predict next day, roll forward. No future leakage. Use full history up to each date for training, then predict the next day.

In [ ]:
def rolling_backtest(price, n_lags, model_factory, min_train=500):
    """model_factory() returns a fitted sklearn pipeline; we refit at each step (or every N steps to save time)."""
    T = len(price)
    preds = []
    actuals = []
    for end in range(min_train + n_lags + 1, T):
        train_price = price[:end]
        X_tr, y_tr = build_lag_features(train_price, n_lags)
        model = model_factory()
        model.fit(X_tr, y_tr)
        # last row = [p(end-2), ..., p(end-1-n_lags)] to match build_lag_features column order
        last_X = train_price[end - 1 - n_lags : end - 1][::-1].reshape(1, -1)
        preds.append(model.predict(last_X)[0])
        actuals.append(price[end])
    return np.array(preds), np.array(actuals)

# Optional: run on a slice of test period to keep runtime reasonable (or refit every N days)
test_start_idx = val_end  # first index of test in full df
price_full = df['price'].values
def make_pipe():
    return Pipeline([
        ('preprocess', ColumnTransformer([('scale', StandardScaler(), list(range(N_LAGS)))], remainder='passthrough')),
        ('regressor', Ridge(alpha=1.0))
    ])
backtest_preds, backtest_actuals = rolling_backtest(price_full, N_LAGS, model_factory=make_pipe)
m_roll = regression_metrics(backtest_actuals, backtest_preds)
print('Rolling backtest metrics:', m_roll)

## 2.4 TimeSeriesSplit + RandomizedSearchCV

Use `TimeSeriesSplit` and `RandomizedSearchCV` to tune Ridge alpha (and/or other hyperparameters). Document param grid.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {'regressor__alpha': np.logspace(-2, 2, 20)}
search = RandomizedSearchCV(
    pipe, param_distributions=param_grid, n_iter=10, cv=tscv,
    scoring='neg_mean_absolute_error', random_state=42
)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print('Best CV MAE:', -search.best_score_)

## 2.5 Evaluation: metrics, residual plots, error discussion

Report rolling backtest MAE, RMSE, directional accuracy; plot residuals; short discussion of where the model performs poorly.

In [ ]:
y_pred_final = search.predict(X_test)
metrics_final = regression_metrics(y_test, y_pred_final)
print('Test set (best model):', metrics_final)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
residuals = y_test - y_pred_final
axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Residuals')
axes[0].set_xlabel('Error')
axes[1].scatter(y_pred_final, y_test, alpha=0.5, s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='y=x')
axes[1].set_title('Predicted vs Actual')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].legend()
plt.tight_layout()
plt.show()